# Phase 1 data: full training set + Tier-B + evaluation set + preselection

**Repo:** `octoborg-x/remove-photo-bg` · **branch:** `feat/distill-phase1-data`
**Spec:** `optimize_model_plan/audit.md` §6-Phase1

This notebook builds the **whole** Phase-1 dataset and stops for review. Run
top to bottom; every cell is resumable (fetchers merge into the manifest by
sha256, so re-running never duplicates and never loses work).

1. Clone the branch that has the fixed fetchers (`manifest_io.py`, `image_ops.py`,
   `commons_api.py`, `fetch-eval.py`, Tier-B categories, leakage guard).
2. Build the **training set**: Cleveland MAA bulk (the ~14k workhorse) + Commons
   topical + Tier-B (hair/fur/soft edges) + Met (best effort).
3. Build the **evaluation set** (`data/eval-manifest.csv`, `split=test`).
4. Run deterministic **preselection** with the eval-set leakage guard.
5. Print real per-source / tier / split / category counts.

No GPU is required (HTTP + PIL + scipy only).

> **Never invent numbers.** Report exactly what the last cell prints.
> The training target is 10,000-30,000 images. If Cleveland is unreachable the
> counts will be far below that - report it, do not pad.


## 1. Runtime check

In [ ]:
import os, sys, shutil
print("Hostname:", os.uname().nodename)
print("Python:", sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "(no GPU - fine, fetching is CPU-only)"
print("Disk:", shutil.disk_usage("/content"))


## 2. Clone the branch (or update an existing clone)

In [ ]:
import os, subprocess

BRANCH = "feat/distill-phase1-data"
REPO_URL = "https://github.com/octoborg-x/remove-photo-bg.git"
REPO = "/content/remove-photo-bg"

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout)


## 3. Dependencies

In [ ]:
%pip install -q pillow numpy scipy requests
import PIL, numpy, scipy, requests
print("deps ok:", PIL.__version__, numpy.__version__, scipy.__version__)


## 4. Training set - Cleveland MAA bulk (the ~14k workhorse)

Licence-gated: only `share_license_status == "CC0"` records are kept.
This is the bulk source; if it returns few rows, tell me before continuing.


In [ ]:
# per-type 3200 exhausts the CC0 object types (Ceramic ~3.0k, Textile ~2.1k,
# Sculpture ~2.0k, Metalwork ~1.4k, ...); ~14k rows, roughly 2-3 h at the
# API's 0.3 s pacing. Re-running is cheap: images already in the manifest
# are skipped by URL, so a resume only downloads what is new.
!cd /content/remove-photo-bg && python3 scripts/distill/fetch-cleveland.py --per-type 3200 --workers 6 2>&1 | tail -6


## 5. Training set - Commons topical (product / jewelry / plant / pet / glass / object)

Topical CC0/PD-heavy categories, licence-checked per image.


In [ ]:
!cd /content/remove-photo-bg && python3 scripts/distill/fetch-cc0.py --categories product,shoe,jewelry,plant,pet,glass,object --count 150 2>&1 | tail -5


## 6. Training set - Tier-B (hair / fur / soft edges)

Closes the gap where museum sources yield no soft-edge rows. Category names
contain *hair*/*fur* so `select.py` tiers them **B**. Disjoint from the eval
categories in section 8.


In [ ]:
!cd /content/remove-photo-bg && python3 scripts/distill/fetch-cc0.py --categories hair,fur --count 150 2>&1 | tail -5


## 7. Training set - Met Museum (best effort)

The Met object endpoint often 403s from shared Colab egress IPs. This cell is
non-fatal: if it fails, the rest still works.


In [ ]:
!cd /content/remove-photo-bg && python3 scripts/distill/fetch-met.py --per-query 80 2>&1 | tail -5 || echo "(Met unavailable - continuing)"


## 8. Evaluation set

Writes `data/eval-manifest.csv` (`split=test`) and `data/eval-images/`. Categories
are disjoint from training. Never trained on or tuned; downstream metrics are
"agreement with teacher, not accuracy" (no ground-truth masks).


In [ ]:
!cd /content/remove-photo-bg && python3 scripts/distill/fetch-eval.py --per-category 15 2>&1 | tail -5


## 9. Manifest integrity check (do not skip)

Confirms the training manifest actually holds the bulk rows before preselection.


In [ ]:
import csv, os
from collections import Counter
rows = list(csv.DictReader(open("data/manifest.csv"))) if os.path.exists("data/manifest.csv") else []
print("training manifest rows:", len(rows))
print("by source:", dict(Counter(r["source"] for r in rows).most_common()))
print("by category:", dict(Counter(r["category"] for r in rows).most_common()))
print("unique sha256:", len({r["sha256"] for r in rows}))
if len(rows) < 10000:
    print("\n*** WARNING: below the 10,000 target. Check that section 4 "
          "(Cleveland) ran; re-run it if it errored. ***")


## 10. Preselection (deterministic) with leakage guard

Rejects: images in the eval set, missing files, short side < 768, low sharpness,
near-duplicates. Writes `data/manifest-selected.csv` and
`data/manifest-rejected.csv`.


In [ ]:
!cd /content/remove-photo-bg && python3 scripts/distill/select.py \
    --manifest data/manifest.csv \
    --out data/manifest-selected.csv \
    --rejected-out data/manifest-rejected.csv 2>&1 | tail -20


## 11. Real counts (report these)

In [ ]:
import csv, json, os
from collections import Counter

def load(p):
    return list(csv.DictReader(open(p))) if os.path.exists(p) else []

raw = load("data/manifest.csv")
sel = load("data/manifest-selected.csv")
rej = load("data/manifest-rejected.csv")
ev  = load("data/eval-manifest.csv")

def grp(rows, key): return dict(Counter(r.get(key,"") for r in rows).most_common())
def reason(s):
    if s.startswith("short-side"): return "short-side"
    if s.startswith("sharpness"): return "sharpness"
    return s

summary = {
    "raw_rows": len(raw),
    "raw_by_source": grp(raw, "source"),
    "raw_by_category": grp(raw, "category"),
    "selected_rows": len(sel),
    "selected_by_split": grp(sel, "split"),
    "selected_by_tier": grp(sel, "tier"),
    "selected_by_category": grp(sel, "category"),
    "rejected_total": len(rej),
    "rejected_by_reason": dict(Counter(reason(r.get("reject_reason","")) for r in rej).most_common()),
    "eval_rows": len(ev),
    "eval_by_category": grp(ev, "category"),
}
print(json.dumps(summary, indent=2))
os.makedirs("smoke-results/distill", exist_ok=True)
json.dump(summary, open("smoke-results/distill/phase1-counts.json","w"), indent=2)
print("\nwrote smoke-results/distill/phase1-counts.json")


## 12. STOP

Phase 1 is complete when the training manifest holds 10,000-30,000 rows and the
eval set is >= 100 images. Report the JSON from cell 11 back for review **before**
any training (spec §6-Phase1.4 STOP).

If a fetcher printed a low number, re-run its cell - it resumes.
